In [1]:
import torch
import torch.nn.functional as F

import pandas as pd

import preprocess
import update_model
import validate

In [111]:
def get_device():
    if torch.backends.mps.is_available():
        print("MPS is available")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("CUDA is available")
        return torch.device("cuda")
    else:
        print("CPU is available")
        return torch.device("cpu")

DEVICE = get_device()
print('Using device:', str(DEVICE).upper(), "\n")

MODEL_DIR="./model"
DATA_PATH = "war_and_peace.txt"

TRAIN_DATA_BUCKET = "cbow-training-data-1f656"
MODEL_DATA_BUCKET = "cbow-model-data-1f656"

MPS is available
Using device: MPS 



In [ ]:
# # === Run this code for the first model initialization ===

# # Download training data
# preprocess.download_data_from_gcs(
#     TRAIN_DATA_BUCKET,
#     DATA_PATH,
# )

# model, cfg = update_model.init_first_model(data_path=DATA_PATH)

In [112]:
# === Run this code to initialize pretrained model ===

# Download training data
preprocess.download_data_from_gcs(
    TRAIN_DATA_BUCKET,
    DATA_PATH
)

# Download pretrained model and its configuration
preprocess.download_model_from_gcs(
    bucket_name=MODEL_DATA_BUCKET,
    download_dir=MODEL_DIR,
)

Data already exists at data/war_and_peace.txt
Latest version folder: version-1
Model already exists at ./model/model_cpu.pt
Model already exists at ./model/model_ns_config.json
Model already exists at ./model/sequence.pt


In [113]:
# === Initialize pretrained model ===
model, cfg = update_model.init_model(
    new_data_path=f"data/{DATA_PATH}"
)

 CBOW Incremental Update
   new data : data/war_and_peace.txt
   device   : MPS
 -- Loaded weights from ./model/model_cpu.pt
 -- Existing vocab size : 58,331
 -- New words added     : 0
 -- Merged vocab size   : 58,331
 -- Sequence length: 3,722,244 tokens  (saved → ./model/sequence.pt)
 -- Vocab unchanged; no layer resizing needed

 -- Done ✓


In [ ]:
# === Train model ===
model = update_model.train(
    model=model,
    cfg=cfg,
    device=DEVICE,
    epochs=100,
    monitor_step=1,
    lr=0.0001
)

# === Save model configuration locally ===
update_model.save(model, cfg)

In [ ]:
# # === Save model configuration in GCS and create new version ===
# preprocess.save_model_to_gcs(bucket_name=MODEL_DATA_BUCKET)

In [121]:
model = model.to(DEVICE)

word_embeddings = model.get_input_embeddings() # Shape: (vocab_size, embedding_dim)
word_embeddings_n = F.normalize(word_embeddings, p=2, dim=1)

window_size = cfg['window_size']
word2idx    = cfg["word2idx"]
idx2word    = {i: w for w, i in word2idx.items()}

In [ ]:
print("\n" + "="*40)

test_word = "red"
top_k = 5
print(f"Top {top_k} similar words to '{test_word}':")
res = validate.find_similar_words(
    test_word, 
    word_embeddings_n, 
    word2idx, 
    idx2word,
    k=top_k
)

for w, score in res:
    print(f"  {w:15} {score:.4f}")
print("="*40)

In [123]:
pairs = [
    ("two",  "three"),    # should be HIGH ~0.7+
    ("red",   "green"),      # should be HIGH ~0.6+
    ("white", "never"),    # should be LOW  ~0.1-0.2
    ("the",   "banana"),   # should be LOW  ~0.1-0.2
    ("sweet",   "sour"),   # should be LOW  ~0.1-0.2
    ("agreement", "old") 
]
for w1, w2 in pairs:
    if w1 in word2idx and w2 in word2idx:
        v1 = word_embeddings_n[word2idx[w1]]
        v2 = word_embeddings_n[word2idx[w2]]
        sim = F.cosine_similarity(v1, v2, dim=0).item()
        print(f"  {w1:10} ~ {w2:10}  →  {sim:.4f}")

  two        ~ three       →  0.7290
  red        ~ green       →  0.5253
  white      ~ never       →  0.0265
  the        ~ banana      →  0.0765
  sweet      ~ sour        →  0.2762
  agreement  ~ old         →  0.1228


In [133]:
words = [
    "vision",
    "color",
    "red",
    "orange",
    "yellow",
    "green",
    "blue",
    "violet",
    "purple",
    "lilac",
    "taste",
    "bitter",
    "sweet",
    "sour"
]

iteration = "iteration-2"

In [125]:
for w in ["red", "orange", "yellow", "green", "blue", "violet", "purple", "lilac"]:
    print(f"{w} --- {word_embeddings[word2idx[w]].norm()}")

red --- 7.948973655700684
orange --- 11.514387130737305
yellow --- 8.437150001525879
green --- 8.817301750183105
blue --- 8.484273910522461
violet --- 5.486354827880859
purple --- 10.520554542541504
lilac --- 12.580647468566895


In [135]:
embedding_size = cfg['embedding_shape'][1]
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

thresholds = [0.2, 0.4, 1, 3, 5, 6, 7 ]
# thresholds = [i for i in range(1, 11)]

for t in thresholds:
    
    # filtering
    B = (embeddings >= t).int()
    # C = torch.cov(B.T)
    C = B @ B.T

    print(f" -- threshold = {t}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/{iteration}/filtration_1-cov_matrix_{t}.csv", mode='x')
    # iteration-2
    # iteration-1

 -- threshold = 0.2, nonzoer(B) = 734
 -- threshold = 0.4, nonzoer(B) = 567
 -- threshold = 1, nonzoer(B) = 221
 -- threshold = 3, nonzoer(B) = 5
 -- threshold = 5, nonzoer(B) = 0
 -- threshold = 6, nonzoer(B) = 0
 -- threshold = 7, nonzoer(B) = 0


In [136]:
embeddings = torch.zeros(len(words), embedding_size)
for i in range(len(embeddings)):
    embeddings[i] = word_embeddings[word2idx[words[i]]]

deltas = [3, 2, 1, 0.5, 0.3, 0.1, 0.05]

for d in deltas:
    # filtering
    B = (torch.abs(embeddings) <= d).int()
    C = B @ B.T

    print(f"-- delta = {d}, nonzoer(B) = {len((B == 1).nonzero())}")

    df = pd.DataFrame(C, index=words, columns=words)

    df.to_csv(f"model_artifacts/{iteration}/filtration_2-cov_matrix_{d}.csv",  mode='x')

-- delta = 3, nonzoer(B) = 1785
-- delta = 2, nonzoer(B) = 1739
-- delta = 1, nonzoer(B) = 1372
-- delta = 0.5, nonzoer(B) = 826
-- delta = 0.3, nonzoer(B) = 511
-- delta = 0.1, nonzoer(B) = 171
-- delta = 0.05, nonzoer(B) = 75
